In [4]:
# ==========================================================
# BASE DE DATOS SUCIA - SIMULACIÓN PRUEBA TÉCNICA FLUID ATTACKS
# Incluye: NULLs, Nulos Implícitos ('N/A', 'UNKNOWN', 'NaN'),
# Duplicados, Inconsistencias de Tipo y Fechas Mal Formateadas.
# ==========================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = SparkSession.builder \
    .appName("FluidAttacks_RealData_Prep") \
    .master("local[*]") \
    .getOrCreate()

# ----------------------------------------------------------
# TABLA 1: proyectos (Con nulos y datos faltantes)
# ----------------------------------------------------------
proyectos_data = [
    (101, "Fintech-API", "Backend", "Alta"),
    (102, "ECommerce-Web", "Frontend", None),        # NULL en criticidad
    (103, "Auth-Service", None, "Crítica"),           # NULL en tipo
    (104, "Data-Pipeline", "Data", "UNKNOWN"),        # Nulo implícito
    (105, "Legacy-Portal", "Web", "Crítica")
]
proyectos_schema = ["id_proyecto", "nombre_proyecto", "tipo", "criticidad_negocio"]

# ----------------------------------------------------------
# TABLA 2: hallazgos (Data Semiautomática de Scanners de Seguridad)
# ----------------------------------------------------------
hallazgos_schema = StructType([
    StructField("id_hallazgo", IntegerType(), True),
    StructField("id_proyecto", IntegerType(), True),
    StructField("vulnerabilidad", StringType(), True),
    StructField("severidad", StringType(), True),
    StructField("cvss_score", DoubleType(), True),    # Nulos explícitos
    StructField("fecha_deteccion", StringType(), True),# Formatos de fecha mixtos
    StructField("fecha_remediacion", StringType(), True),
    StructField("estado", StringType(), True)
])

hallazgos_data = [
    # Normales y resueltos
    (1, 101, "SQL Injection", "CRITICAL", 9.8, "2026-01-10", "2026-01-12", "REMEDIATED"),
    # Sin fecha de remediación ni CVSS (Abierta)
    (2, 101, "XSS Cross-Site Scripting", "HIGH", None, "2026-01-15", None, "OPEN"),
    # Nulo implícito como String ('NULL' o 'N/A')
    (3, 101, "Hardcoded Credentials", "CRITICAL", 8.9, "2026-02-01", "N/A", "REMEDIATED"),
    # Formato de fecha distinto (MM/DD/YYYY)
    (4, 102, "Insecure Dependency", "MEDIUM", 5.3, "01/20/2026", "02/15/2026", "REMEDIATED"),
    # CVSS Nulo
    (5, 102, "XSS Cross-Site Scripting", "MEDIUM", None, "2026-02-10", None, "OPEN"),
    # SLA ultra rápido (0 días)
    (6, 103, "Remote Code Execution", "CRITICAL", 10.0, "2026-01-05", "2026-01-05", "REMEDIATED"),
    # Sin severidad reportada
    (7, 103, "Broken Access Control", None, 8.1, "2026-01-12", None, "IN_PROGRESS"),
    # Duplicado exacto del ID 1 (Filtro requerido)
    (1, 101, "SQL Injection", "CRITICAL", 9.8, "2026-01-10", "2026-01-12", "REMEDIATED"),
    # Duplicado lógico (Diferente ID, mismo evento registrado 2 veces)
    (13, 101, "SQL Injection", "CRITICAL", 9.8, "2026-01-10", "2026-01-12", "REMEDIATED"),
    # Outlier en fecha (Fecha futura o errónea de logs)
    (8, 104, "CSRF", "LOW", 3.2, "2030-01-01", None, "OPEN"),
    # Sin proyecto asignado (Orfano - probar RIGHT/FULL JOINs)
    (9, 999, "Unauthenticated Access", "HIGH", 8.7, "2026-02-01", None, "OPEN")
]

# ----------------------------------------------------------
# TABLA 3: logs_autenticacion (Inyecciones de Brute Force)
# ----------------------------------------------------------
logs_data = [
    (1001, "dev_user1", "192.168.1.10", "SUCCESS", "2026-02-20 08:30:00"),
    (1002, "admin_sec", None, "SUCCESS", "2026-02-20 08:32:00"),             # IP Nula
    (1003, "attacker_x", "45.33.22.11", "FAILED", "2026-02-20 08:35:00"),
    (1004, "attacker_x", "45.33.22.11", "FAILED", "2026-02-20 08:35:10"),
    (1005, "attacker_x", "45.33.22.11", "FAILED", "2026-02-20 08:35:20"),
    (1006, "attacker_x", "45.33.22.11", "SUCCESS", "2026-02-20 08:36:00"),  # Ataque exitoso
    (1007, None, "185.220.101.5", "FAILED", "2026-02-20 09:15:00")            # Usuario Nulo
]
logs_schema = ["id_log", "usuario", "ip_origen", "resultado", "timestamp_acceso"]

# Crear DataFrames
df_proyectos = spark.createDataFrame(proyectos_data, proyectos_schema)
df_hallazgos = spark.createDataFrame(hallazgos_data, hallazgos_schema)
df_logs = spark.createDataFrame(logs_data, logs_schema)

# Crear Vistas SQL
df_proyectos.createOrReplaceTempView("proyectos")
df_hallazgos.createOrReplaceTempView("hallazgos")
df_logs.createOrReplaceTempView("logs_autenticacion")

print("¡Base de Datos REALISTA cargada!")    

¡Base de Datos REALISTA cargada!


In [5]:
query = """
SELECT *
FROM proyectos
"""
spark.sql(query).show()

+-----------+---------------+--------+------------------+
|id_proyecto|nombre_proyecto|    tipo|criticidad_negocio|
+-----------+---------------+--------+------------------+
|        101|    Fintech-API| Backend|              Alta|
|        102|  ECommerce-Web|Frontend|              NULL|
|        103|   Auth-Service|    NULL|           Crítica|
|        104|  Data-Pipeline|    Data|           UNKNOWN|
|        105|  Legacy-Portal|     Web|           Crítica|
+-----------+---------------+--------+------------------+



In [6]:
query = """
SELECT *
FROM hallazgos 
"""
spark.sql(query).show()

+-----------+-----------+--------------------+---------+----------+---------------+-----------------+-----------+
|id_hallazgo|id_proyecto|      vulnerabilidad|severidad|cvss_score|fecha_deteccion|fecha_remediacion|     estado|
+-----------+-----------+--------------------+---------+----------+---------------+-----------------+-----------+
|          1|        101|       SQL Injection| CRITICAL|       9.8|     2026-01-10|       2026-01-12| REMEDIATED|
|          2|        101|XSS Cross-Site Sc...|     HIGH|      NULL|     2026-01-15|             NULL|       OPEN|
|          3|        101|Hardcoded Credent...| CRITICAL|       8.9|     2026-02-01|              N/A| REMEDIATED|
|          4|        102| Insecure Dependency|   MEDIUM|       5.3|     01/20/2026|       02/15/2026| REMEDIATED|
|          5|        102|XSS Cross-Site Sc...|   MEDIUM|      NULL|     2026-02-10|             NULL|       OPEN|
|          6|        103|Remote Code Execu...| CRITICAL|      10.0|     2026-01-05|     

In [7]:
query = """
SELECT *
FROM logs_autenticacion
"""
spark.sql(query).show()

+------+----------+-------------+---------+-------------------+
|id_log|   usuario|    ip_origen|resultado|   timestamp_acceso|
+------+----------+-------------+---------+-------------------+
|  1001| dev_user1| 192.168.1.10|  SUCCESS|2026-02-20 08:30:00|
|  1002| admin_sec|         NULL|  SUCCESS|2026-02-20 08:32:00|
|  1003|attacker_x|  45.33.22.11|   FAILED|2026-02-20 08:35:00|
|  1004|attacker_x|  45.33.22.11|   FAILED|2026-02-20 08:35:10|
|  1005|attacker_x|  45.33.22.11|   FAILED|2026-02-20 08:35:20|
|  1006|attacker_x|  45.33.22.11|  SUCCESS|2026-02-20 08:36:00|
|  1007|      NULL|185.220.101.5|   FAILED|2026-02-20 09:15:00|
+------+----------+-------------+---------+-------------------+



---

In [30]:
query = """
SELECT *
FROM hallazgos AS ha
"""
query = """
WITH c_hallazgos AS(
    SELECT DISTINCT *
    FROM hallazgos
),
ha_count AS(
    SELECT
        ha.vulnerabilidad AS vuln,
        COUNT(ha.vulnerabilidad) OVER(
            PARTITION BY ha.vulnerabilidad
        ) AS vuln_count
    FROM c_hallazgos AS ha
    ORDER BY
        vuln_count DESC
)

SELECT
    vuln,
    vuln_count,
    ROW_NUMBER() OVER(
        PARTITION BY vuln
        ORDER  BY vuln_count DESC
    ) AS rank
FROM ha_count
ORDER BY
    vuln_count DESC
"""
spark.sql(query).show()

+--------------------+----------+----+
|                vuln|vuln_count|rank|
+--------------------+----------+----+
|       SQL Injection|         2|   1|
|       SQL Injection|         2|   2|
|XSS Cross-Site Sc...|         2|   1|
|XSS Cross-Site Sc...|         2|   2|
|Broken Access Con...|         1|   1|
|                CSRF|         1|   1|
|Hardcoded Credent...|         1|   1|
| Insecure Dependency|         1|   1|
|Remote Code Execu...|         1|   1|
|Unauthenticated A...|         1|   1|
+--------------------+----------+----+



In [ ]:
query = """
SELECT
    id_proyecto,
    vulnerabilidad,
    COALESCE(cvss_score, 0.0) AS cvss_score_limpio,
    DENSE_RANK() OVER(
        PARTITION BY id_proyecto 
        ORDER BY COALESCE(cvss_score, 0.0) DESC
    ) AS rank_vulnerabilidad
FROM hallazgos
"""
spark.sql(query).show()

+-----------+--------------------+-----------------+-------------------+
|id_proyecto|      vulnerabilidad|cvss_score_limpio|rank_vulnerabilidad|
+-----------+--------------------+-----------------+-------------------+
|        101|       SQL Injection|              9.8|                  1|
|        101|       SQL Injection|              9.8|                  1|
|        101|       SQL Injection|              9.8|                  1|
|        101|Hardcoded Credent...|              8.9|                  2|
|        101|XSS Cross-Site Sc...|              0.0|                  3|
|        102| Insecure Dependency|              5.3|                  1|
|        102|XSS Cross-Site Sc...|              0.0|                  2|
|        103|Remote Code Execu...|             10.0|                  1|
|        103|Broken Access Con...|              8.1|                  2|
|        104|                CSRF|              3.2|                  1|
|        999|Unauthenticated A...|              8.7

In [13]:
query = """
SELECT
    id_proyecto,
    vulnerabilidad,
    COALESCE(cvss_score, 0.0) AS cvss_score_limpio,
    DENSE_RANK() OVER(
        PARTITION BY id_proyecto 
        ORDER BY COALESCE(cvss_score, 0.0) DESC
    ) AS rank_vulnerabilidad,
    MAX(COALESCE(cvss_score, 0.0)) OVER(
        PARTITION BY id_proyecto
        ORDER BY COALESCE(cvss_score, 0.0) DESC
    ) AS max_vulberabilidad
FROM hallazgos
"""
spark.sql(query).show()

+-----------+--------------------+-----------------+-------------------+------------------+
|id_proyecto|      vulnerabilidad|cvss_score_limpio|rank_vulnerabilidad|max_vulberabilidad|
+-----------+--------------------+-----------------+-------------------+------------------+
|        101|       SQL Injection|              9.8|                  1|               9.8|
|        101|       SQL Injection|              9.8|                  1|               9.8|
|        101|       SQL Injection|              9.8|                  1|               9.8|
|        101|Hardcoded Credent...|              8.9|                  2|               9.8|
|        101|XSS Cross-Site Sc...|              0.0|                  3|               9.8|
|        102| Insecure Dependency|              5.3|                  1|               5.3|
|        102|XSS Cross-Site Sc...|              0.0|                  2|               5.3|
|        103|Remote Code Execu...|             10.0|                  1|        

### Ejercicio 15 — Fechas mixtas y cálculo de SLA
Objetivo: Normaliza fecha_deteccion (que viene en dos formatos: 'YYYY-MM-DD' y 'MM/DD/YYYY') a un tipo fecha real usando CASE + funciones de conversión, y luego calcula cuántos días pasaron entre detección y remediación para los hallazgos ya remediados. Conceptos: CASE WHEN combinado con TO_DATE/to_date y un patrón de formato, DATEDIFF, JOIN opcional con proyectos para mostrar el nombre del proyecto junto al SLA. Pista: antes de escribir la conversión, identifica con una consulta exploratoria cuáles filas tienen cada formato (por ejemplo, buscando el patrón / en el string). No asumas el formato — verifícalo con datos.

In [9]:
query = """
WITH 
f_hallazgos AS (
    SELECT DISTINCT *
    FROM hallazgos AS ha
),
fechas AS (
    SELECT
        ha.id_hallazgo,
        -- Retorna tipo DATE puro
        COALESCE(
            TRY_TO_DATE(ha.fecha_deteccion, 'yyyy-MM-dd'),
            TRY_TO_DATE(ha.fecha_deteccion, 'MM/dd/yyyy'),
            TRY_TO_DATE(ha.fecha_deteccion, 'dd/MM/yyyy')
        ) AS fecha_deteccion_dt,

        -- Retorna tipo DATE puro
        COALESCE(
            TRY_TO_DATE(ha.fecha_remediacion, 'yyyy-MM-dd'),
            TRY_TO_DATE(ha.fecha_remediacion, 'MM/dd/yyyy'),
            TRY_TO_DATE(ha.fecha_remediacion, 'dd/MM/yyyy'),
            DATE '0001-01-01'
        ) AS fecha_remediacion_dt
    FROM hallazgos AS ha
) 

SELECT
    f_ha.id_hallazgo,
    -- Se le da formato visual usando los patrones correctos en minúscula
    DATE_FORMAT(fh.fecha_deteccion_dt, 'dd/MM/yyyy') AS deteccion,
    DATE_FORMAT(fh.fecha_remediacion_dt, 'dd/MM/yyyy') AS remediacion,
    -- DATEDIFF recibe directamente las variables de tipo DATE
    DATEDIFF(fh.fecha_remediacion_dt, fh.fecha_deteccion_dt) AS dias_diferencia
FROM f_hallazgos AS f_ha
INNER JOIN fechas AS fh
    ON f_ha.id_hallazgo = fh.id_hallazgo
"""
spark.sql(query).show()

+-----------+----------+-----------+---------------+
|id_hallazgo| deteccion|remediacion|dias_diferencia|
+-----------+----------+-----------+---------------+
|          1|10/01/2026| 12/01/2026|              2|
|          2|15/01/2026| 01/01/0001|        -739630|
|          3|01/02/2026| 01/01/0001|        -739647|
|          4|20/01/2026| 15/02/2026|             26|
|          5|10/02/2026| 01/01/0001|        -739656|
|          6|05/01/2026| 05/01/2026|              0|
|          7|12/01/2026| 01/01/0001|        -739627|
|          1|10/01/2026| 12/01/2026|              2|
|         13|10/01/2026| 12/01/2026|              2|
|          8|01/01/2030| 01/01/0001|        -741077|
|          9|01/02/2026| 01/01/0001|        -739647|
+-----------+----------+-----------+---------------+



In [10]:
query = """
WITH 
f_hallazgos AS (
    SELECT DISTINCT *
    FROM hallazgos AS ha
),
fechas AS (
    SELECT
        ha.id_hallazgo,
        -- Conversión de fecha de detección
        DATE_FORMAT(
        IFNULL(
            COALESCE(
                TRY_TO_DATE(ha.fecha_deteccion, 'yyyy-MM-dd'),
                TRY_TO_DATE(ha.fecha_deteccion, 'MM/dd/yyyy'),
                TRY_TO_DATE(ha.fecha_deteccion, 'dd/MM/yyyy')
            ), DATE '0001-01-01'
            ), 'dd/MM/yyyy') AS fecha_deteccion_formato,

        -- Conversión de fecha de remediación con fecha por defecto para nulos
        DATE_FORMAT(
            COALESCE(
                TRY_TO_DATE(ha.fecha_remediacion, 'yyyy-MM-dd'),
                TRY_TO_DATE(ha.fecha_remediacion, 'MM/dd/yyyy'),
                TRY_TO_DATE(ha.fecha_remediacion, 'dd/MM/yyyy'),
                DATE '0001-01-01'  -- Corrección: Formato ISO YYYY-MM-DD
            ),
            'dd/MM/yyyy'           -- Corrección: Se agrega el formato de salida
        ) AS fecha_remediacion_formato
    FROM hallazgos AS ha
) 

SELECT
    f_ha.id_hallazgo,
    fh.fecha_deteccion_formato,
    fh.fecha_remediacion_formato
FROM f_hallazgos AS f_ha
INNER JOIN fechas AS fh
    ON f_ha.id_hallazgo = fh.id_hallazgo
"""
spark.sql(query).show()

+-----------+-----------------------+-------------------------+
|id_hallazgo|fecha_deteccion_formato|fecha_remediacion_formato|
+-----------+-----------------------+-------------------------+
|          1|             10/01/2026|               12/01/2026|
|          2|             15/01/2026|               01/01/0001|
|          3|             01/02/2026|               01/01/0001|
|          4|             20/01/2026|               15/02/2026|
|          5|             10/02/2026|               01/01/0001|
|          6|             05/01/2026|               05/01/2026|
|          7|             12/01/2026|               01/01/0001|
|          1|             10/01/2026|               12/01/2026|
|         13|             10/01/2026|               12/01/2026|
|          8|             01/01/2030|               01/01/0001|
|          9|             01/02/2026|               01/01/0001|
+-----------+-----------------------+-------------------------+



### Ejercicio 14 — Deduplicación exacta y lógica
Objetivo: Identifica primero los duplicados exactos (mismo id_hallazgo repetido con todos los demás campos iguales) y luego, por separado, los duplicados lógicos (mismo id_proyecto, vulnerabilidad, fecha_deteccion y severidad, pero con id_hallazgo distinto). Conceptos: GROUP BY sobre todas las columnas + HAVING COUNT(*) > 1 para exactos; GROUP BY sobre un subconjunto de columnas de negocio para lógicos; diferencia conceptual entre ambos tipos de duplicado. Pista: un DISTINCT simple resuelve el duplicado exacto, pero no el lógico — el lógico requiere que definas tú qué combinación de columnas identifica "el mismo evento". Esa decisión de negocio es tan importante como la consulta misma.

In [11]:
query = """
SELECT
    id_hallazgo,
    id_proyecto,
    vulnerabilidad,
    severidad,
    COUNT(*)
FROM hallazgos AS ha
GROUP BY
    id_hallazgo,
    id_proyecto,
    vulnerabilidad,
    severidad
HAVING
    COUNT(*) > 1
"""
spark.sql(query).show()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "c:\Users\Hugo\anaconda3\envs\fraud-analytics\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Hugo\anaconda3\envs\fraud-analytics\Lib\site-packages\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Hugo\anaconda3\envs\fraud-analytics\Lib\socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
query = """
SELECT
    id_proyecto,
    vulnerabilidad,
    severidad,
    COUNT(*)
FROM hallazgos AS ha
GROUP BY
    id_proyecto,
    vulnerabilidad,
    severidad
HAVING
    COUNT(*) > 1
"""
spark.sql(query).show()

+-----------+--------------+---------+--------+
|id_proyecto|vulnerabilidad|severidad|count(1)|
+-----------+--------------+---------+--------+
|        101| SQL Injection| CRITICAL|       3|
+-----------+--------------+---------+--------+



### Ejercicio 13 — Detectar registros huérfanos con RIGHT/FULL JOIN
Objetivo: Encuentra los hallazgos que hacen referencia a un id_proyecto que no existe en la tabla proyectos. Conceptos: RIGHT JOIN o FULL JOIN, filtro WHERE proyectos.id_proyecto IS NULL para aislar los huérfanos. Pista: esto conecta directamente con lo que anotaste en el Ejercicio 11. En un contexto de seguridad, un hallazgo sin proyecto válido es exactamente el tipo de inconsistencia que un analista debe reportar, no ignorar.

In [ ]:
query = """
SELECT
    *
FROM proyectos AS py
FULL JOIN hallazgos AS ha
    ON py.id_proyecto = ha.id_proyecto
"""
spark.sql(query).show()

+-----------+---------------+--------+------------------+-----------+-----------+--------------------+---------+----------+---------------+-----------------+-----------+
|id_proyecto|nombre_proyecto|    tipo|criticidad_negocio|id_hallazgo|id_proyecto|      vulnerabilidad|severidad|cvss_score|fecha_deteccion|fecha_remediacion|     estado|
+-----------+---------------+--------+------------------+-----------+-----------+--------------------+---------+----------+---------------+-----------------+-----------+
|        101|    Fintech-API| Backend|              Alta|          1|        101|       SQL Injection| CRITICAL|       9.8|     2026-01-10|       2026-01-12| REMEDIATED|
|        101|    Fintech-API| Backend|              Alta|          2|        101|XSS Cross-Site Sc...|     HIGH|      NULL|     2026-01-15|             NULL|       OPEN|
|        101|    Fintech-API| Backend|              Alta|          3|        101|Hardcoded Credent...| CRITICAL|       8.9|     2026-02-01|           

In [ ]:
query = """
SELECT
    *
FROM proyectos AS py
FULL JOIN hallazgos AS ha
    ON py.id_proyecto = ha.id_proyecto
WHERE
    py.id_proyecto IS NULL
"""
spark.sql(query).show()

+-----------+---------------+----+------------------+-----------+-----------+--------------------+---------+----------+---------------+-----------------+------+
|id_proyecto|nombre_proyecto|tipo|criticidad_negocio|id_hallazgo|id_proyecto|      vulnerabilidad|severidad|cvss_score|fecha_deteccion|fecha_remediacion|estado|
+-----------+---------------+----+------------------+-----------+-----------+--------------------+---------+----------+---------------+-----------------+------+
|       NULL|           NULL|NULL|              NULL|          9|        999|Unauthenticated A...|     HIGH|       8.7|     2026-02-01|             NULL|  OPEN|
+-----------+---------------+----+------------------+-----------+-----------+--------------------+---------+----------+---------------+-----------------+------+



### Ejercicio 12 — LEFT JOIN para encontrar ausencias
Objetivo: Lista todos los proyectos, incluyendo los que no tienen ningún hallazgo registrado, junto con el conteo de hallazgos por proyecto (0 si no tiene ninguno). Conceptos: LEFT JOIN, COUNT sobre columna de la tabla derecha (no COUNT(*)), GROUP BY, COALESCE opcional para forzar el 0. Pista: si usas COUNT(*) en vez de COUNT(id_hallazgo), el proyecto sin hallazgos te va a dar 1 en vez de 0. Entender por qué es la esencia de este ejercicio.

In [ ]:
query = """
SELECT
    py.nombre_proyecto AS proyecto,
    COUNT(COALESCE(ha.id_hallazgo, 0)) AS count
FROM proyectos AS py
LEFT JOIN hallazgos AS ha
    ON py.id_proyecto = ha.id_proyecto
GROUP BY
    py.nombre_proyecto
ORDER BY
    count
"""
spark.sql(query).show()

+-------------+-----+
|     proyecto|count|
+-------------+-----+
|Data-Pipeline|    1|
|Legacy-Portal|    1|
|ECommerce-Web|    2|
| Auth-Service|    2|
|  Fintech-API|    5|
+-------------+-----+

